# Phase 1 — Validation chống leakage

Notebook này **chỉ thực hiện Phase 1**, chưa huấn luyện mô hình. Mục tiêu là tạo một bộ 3 fold cố định, đáng tin cậy để mọi thí nghiệm sau được so sánh công bằng bằng OOF Macro F1.

## Phase 1 dùng để làm gì?

Validation mô phỏng việc mô hình gặp dữ liệu chưa từng thấy. Nếu cách chia sai, điểm validation có thể cao nhưng leaderboard thấp. Dữ liệu này có hai rủi ro cụ thể:

1. Train đang được xếp thành các khối theo `track_genre`, vì vậy không được chia theo vị trí dòng.
2. Một số bài hát có 15 feature giống hệt nhau. Nếu hai bản sao nằm ở cả train fold và validation fold, mô hình có thể ghi nhớ mẫu và tạo điểm số lạc quan giả.

Ta dùng `StratifiedGroupKFold` để đồng thời:

- **Stratified**: giữ tỷ lệ 112 genre gần giống nhau giữa các fold.
- **Group**: giữ mọi dòng có bộ 15 feature giống nhau trong cùng một fold.

Đầu ra là `artifacts/validation_folds.csv`. Từ Phase 2 trở đi, mọi model phải dùng đúng file này.

## 0. Chuẩn bị dữ liệu trên Google Colab

Khi chạy cell đầu tiên trên Colab, notebook sẽ mở hộp thoại yêu cầu upload đúng ba file: `train.csv`, `test.csv` và `genre_mapping.csv`. Hãy giữ nguyên tên file. `sample_submission.csv` chưa cần cho Phase 1. Khi chạy local, notebook tự đọc dữ liệu trong thư mục dự án và không hiện hộp thoại upload.

In [1]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42
N_SPLITS = 3
TARGET = "track_genre"
ID_COLUMN = "track_id"
EXPECTED_LABELS = np.arange(112)
REQUIRED_FILES = ("train.csv", "test.csv", "genre_mapping.csv")

# Colab không có sẵn dữ liệu local, nên yêu cầu người dùng upload.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

if IN_COLAB:
    PROJECT_DIR = Path("/content")
    missing_files = [name for name in REQUIRED_FILES if not (PROJECT_DIR / name).exists()]
    if missing_files:
        print("Hãy upload các file còn thiếu:", missing_files)
        colab_files.upload()
    missing_files = [name for name in REQUIRED_FILES if not (PROJECT_DIR / name).exists()]
    if missing_files:
        raise FileNotFoundError(
            f"Thiếu {missing_files}. Hãy upload lại và giữ nguyên tên file."
        )
else:
    # Chạy local từ thư mục dự án hoặc từ workspace root.
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "train.csv").exists():
        PROJECT_DIR = PROJECT_DIR / "ISE_TRAINNING_TEST_23-8-2026"
PROJECT_DIR = PROJECT_DIR.resolve()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

print(f"Python:       {platform.python_version()}")
print(f"Executable:   {Path(__import__('sys').executable)}")
print(f"pandas:       {pd.__version__}")
print(f"scikit-learn:{sklearn.__version__}")
print(f"Environment:  {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Data folder:  {PROJECT_DIR}")

Python:       3.14.4
Executable:   /home/drago/projects/Machine-Learning-Deep-Learning/.venv/bin/python
pandas:       3.0.5
scikit-learn:1.9.0
Environment:  Local
Data folder:  /home/drago/projects/Machine-Learning-Deep-Learning/ISE_TRAINNING_TEST_23-8-2026


## 1. Đọc dữ liệu và khóa schema

`track_id` chỉ dùng để nối kết quả, tuyệt đối không đưa vào model. `FEATURES` được suy ra từ các cột test còn lại để tránh vô tình đưa target vào đầu vào.

In [2]:
train = pd.read_csv(PROJECT_DIR / "train.csv")
test = pd.read_csv(PROJECT_DIR / "test.csv")
genre_mapping = pd.read_csv(PROJECT_DIR / "genre_mapping.csv")

FEATURES = [column for column in test.columns if column != ID_COLUMN]

assert len(FEATURES) == 15, f"Mong đợi 15 features, nhận được {len(FEATURES)}"
assert TARGET in train.columns and TARGET not in test.columns
assert ID_COLUMN not in FEATURES and TARGET not in FEATURES
assert set(train.columns) == {ID_COLUMN, TARGET, *FEATURES}
assert set(test.columns) == {ID_COLUMN, *FEATURES}

display(pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "unique_track_id": [train[ID_COLUMN].nunique(), test[ID_COLUMN].nunique()],
}))
print("FEATURES =", FEATURES)

,dataset,rows,columns,unique_track_id
0,train,51452,17,51452
1,test,21947,16,21947


FEATURES = ['popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']


## 2. Kiểm tra dữ liệu trước khi chia fold

Các assertion làm notebook dừng ngay nếu dữ liệu đầu vào không còn đúng với giả định của dự án. Đây là cách tránh âm thầm tạo fold sai sau khi file dữ liệu thay đổi.

In [3]:
assert train[ID_COLUMN].is_unique, "train có track_id trùng"
assert test[ID_COLUMN].is_unique, "test có track_id trùng"
assert set(train[ID_COLUMN]).isdisjoint(set(test[ID_COLUMN])), "track_id giao nhau giữa train/test"
assert not train.isna().any().any() and not test.isna().any().any(), "Phát hiện NaN"
assert np.isfinite(train[FEATURES].to_numpy(dtype=float)).all(), "Train có infinity"
assert np.isfinite(test[FEATURES].to_numpy(dtype=float)).all(), "Test có infinity"
assert np.array_equal(np.sort(train[TARGET].unique()), EXPECTED_LABELS), "Target không đúng 0..111"
assert np.array_equal(np.sort(genre_mapping["genre_id"].unique()), EXPECTED_LABELS)

class_counts = train[TARGET].value_counts().sort_index()
display(class_counts.describe().to_frame("class_count").T)
print("PASS — ID, NaN/inf, target và genre mapping đều hợp lệ.")

,count,mean,std,min,25%,50%,75%,max
class_count,112.0,459.392857,188.575378,51.0,316.75,488.0,619.75,700.0


PASS — ID, NaN/inf, target và genre mapping đều hợp lệ.


## 3. Tạo `group_id` từ toàn bộ 15 feature

Hai dòng có feature giống hệt nhau nhận cùng hash. Ta còn kiểm tra collision: nếu hai bộ feature khác nhau vô tình có cùng hash, notebook sẽ dừng thay vì coi chúng là một nhóm. `group_id` chỉ phục vụ chia fold, không phải feature của model.

In [4]:
group_id = pd.util.hash_pandas_object(train[FEATURES], index=False).astype("uint64")
group_sizes = group_id.value_counts()

# Trong mỗi hash group, mỗi feature phải chỉ có một giá trị duy nhất.
max_unique_values = train.groupby(group_id, sort=False)[FEATURES].nunique(dropna=False).max(axis=1)
assert max_unique_values.le(1).all(), "Phát hiện hash collision giữa các bộ feature khác nhau"

duplicate_groups = group_sizes[group_sizes > 1]
duplicate_report = pd.DataFrame({
    "unique_feature_groups": [group_id.nunique()],
    "groups_with_duplicates": [len(duplicate_groups)],
    "rows_in_duplicate_groups": [int(duplicate_groups.sum())],
    "duplicate_rows_beyond_first": [int(len(train) - group_id.nunique())],
})
display(duplicate_report)
print("PASS — không có hash collision.")

,unique_feature_groups,groups_with_duplicates,rows_in_duplicate_groups,duplicate_rows_beyond_first
0,50049,585,1988,1403


PASS — không có hash collision.


## 4. Tạo 3 fold cố định

Trong mỗi vòng ở các phase sau, hai fold sẽ dùng để train và fold còn lại để validation. Sau ba vòng, mỗi dòng có đúng một dự đoán **out-of-fold (OOF)** — tức dự đoán từ model chưa từng train trên dòng đó.

In [5]:
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
fold_assignment = np.full(len(train), -1, dtype=np.int8)
split_indices = []

for fold, (train_idx, valid_idx) in enumerate(
    splitter.split(train[FEATURES], train[TARGET], groups=group_id)
):
    assert np.all(fold_assignment[valid_idx] == -1), "Một dòng được gán validation nhiều lần"
    fold_assignment[valid_idx] = fold
    split_indices.append((train_idx, valid_idx))

assert np.all(fold_assignment >= 0), "Có dòng chưa được gán fold"
assert set(fold_assignment.tolist()) == set(range(N_SPLITS))
print("Fold sizes:", dict(zip(*np.unique(fold_assignment, return_counts=True))))

Fold sizes: {np.int8(0): np.int64(17152), np.int8(1): np.int64(17150), np.int8(2): np.int64(17150)}


## 5. Audit leakage và độ cân bằng

Một split đạt yêu cầu khi validation đủ 112 lớp và không có `group_id` nào đồng thời xuất hiện ở train/validation. Ngoài ra ta đo tỷ lệ của từng lớp rơi vào từng fold; lý tưởng xấp xỉ 1/3.

In [6]:
audit_rows = []
for fold, (train_idx, valid_idx) in enumerate(split_indices):
    train_groups = set(group_id.iloc[train_idx])
    valid_groups = set(group_id.iloc[valid_idx])
    group_overlap = len(train_groups.intersection(valid_groups))
    valid_labels = np.sort(train.iloc[valid_idx][TARGET].unique())

    assert group_overlap == 0, f"Fold {fold} bị group leakage"
    assert np.array_equal(valid_labels, EXPECTED_LABELS), f"Fold {fold} thiếu lớp"

    audit_rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "valid_classes": len(valid_labels),
        "train_groups": len(train_groups),
        "valid_groups": len(valid_groups),
        "group_overlap": group_overlap,
        "status": "PASS",
    })

fold_audit = pd.DataFrame(audit_rows).set_index("fold")
display(fold_audit)

# Kiểm tra toàn cục: một group chỉ thuộc đúng một fold.
group_fold_counts = pd.DataFrame({"group_id": group_id, "fold": fold_assignment}).groupby("group_id")["fold"].nunique()
assert group_fold_counts.max() == 1

class_by_fold = pd.crosstab(train[TARGET], fold_assignment)
class_share_by_fold = class_by_fold.div(class_by_fold.sum(axis=1), axis=0)
max_abs_deviation = float((class_share_by_fold - 1 / N_SPLITS).abs().to_numpy().max())

display(class_share_by_fold.describe().T.rename_axis("fold"))
print(f"Sai lệch lớn nhất khỏi tỷ lệ lý tưởng 1/3: {max_abs_deviation:.4%}")
print("PASS — mọi dòng đúng một fold, đủ 112 lớp/fold và không có group leakage.")

,train_rows,valid_rows,valid_classes,train_groups,valid_groups,group_overlap,status
fold,,,,,,,
0,34300,17152,112,33367,16682,0,PASS
1,34302,17150,112,33365,16684,0,PASS
2,34302,17150,112,33366,16683,0,PASS


,count,mean,std,min,25%,50%,75%,max
fold,,,,,,,,
0,112.0,0.333419,0.001591,0.329412,0.332776,0.333333,0.333879,0.344262
1,112.0,0.333254,0.001456,0.327869,0.332704,0.333333,0.333844,0.339286
2,112.0,0.333326,0.001462,0.326733,0.332741,0.333333,0.333879,0.341176


Sai lệch lớn nhất khỏi tỷ lệ lý tưởng 1/3: 1.0929%
PASS — mọi dòng đúng một fold, đủ 112 lớp/fold và không có group leakage.


### Đọc bảng audit

- `valid_rows`: số dòng sẽ được dự đoán OOF trong vòng đó.
- `valid_classes = 112`: mọi genre đều được đánh giá ở mọi fold.
- `group_overlap = 0`: không có bản sao feature lọt qua biên train/validation.
- Sai lệch tỷ lệ lớp nhỏ cho thấy stratification hoạt động hợp lý.

## 6. Lưu artifact dùng chung

CSV chỉ chứa `track_id,fold` để nhẹ và dễ nối lại với train gốc. Trước khi ghi, ta đọc artifact hiện có (nếu có) và từ chối ghi đè nếu cùng `track_id` nhưng fold khác — một chốt an toàn giúp các thí nghiệm không vô tình dùng hai cách chia khác nhau.

In [7]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
fold_path = ARTIFACT_DIR / "validation_folds.csv"
fold_artifact = pd.DataFrame({
    ID_COLUMN: train[ID_COLUMN],
    "fold": fold_assignment.astype(int),
})

if fold_path.exists():
    existing = pd.read_csv(fold_path)
    assert existing.equals(fold_artifact), (
        "Artifact hiện có khác kết quả tái tạo. Không ghi đè; hãy điều tra trước."
    )
    print(f"Artifact đã tồn tại và khớp hoàn toàn: {fold_path}")
else:
    fold_artifact.to_csv(fold_path, index=False)
    print(f"Đã lưu: {fold_path}")

# Read-back validation
saved_folds = pd.read_csv(fold_path)
assert saved_folds.equals(fold_artifact)
assert saved_folds[ID_COLUMN].is_unique
assert len(saved_folds) == len(train)
display(saved_folds.head())

Artifact đã tồn tại và khớp hoàn toàn: /home/drago/projects/Machine-Learning-Deep-Learning/ISE_TRAINNING_TEST_23-8-2026/artifacts/validation_folds.csv


,track_id,fold
0,1iJBSr7s7jYXzM8EGcbK5b,1
1,5vjLSffimiIP26QG5WcN2K,0
2,4mzP5mHkRvGxdhdGdAH7EJ,0
3,4LbWtBkN82ZRhz9jqzgrb3,0
4,3S0OXQeoh0w6AY8WQVckRW,0


## 7. Định nghĩa metric duy nhất

Macro F1 tính F1 riêng cho từng genre rồi lấy trung bình **đều** trên 112 genre. Vì vậy lớp hiếm có tiếng nói ngang lớp lớn. Việc cố định `labels=0..111` bảo đảm một model bỏ sót hoàn toàn một lớp vẫn nhận F1 bằng 0 cho lớp đó, thay vì lớp bị biến mất khỏi phép tính.

In [8]:
def macro_f1(y_true, y_pred):
    """Macro F1 cố định trên đủ 112 nhãn của cuộc thi."""
    return f1_score(
        y_true,
        y_pred,
        labels=EXPECTED_LABELS,
        average="macro",
        zero_division=0,
    )

# Smoke test: dự đoán hoàn hảo trên đủ 112 lớp phải có Macro F1 = 1.
assert macro_f1(EXPECTED_LABELS, EXPECTED_LABELS) == 1.0
print("PASS — macro_f1 đã khóa labels=0..111 và zero_division=0.")

PASS — macro_f1 đã khóa labels=0..111 và zero_division=0.


## 8. Tải artifact về máy khi dùng Colab

Bộ nhớ `/content` của Colab là tạm thời. Cell dưới sẽ tải `validation_folds.csv` về máy để bạn giữ lại và upload lại cho các phase sau.

In [9]:
if IN_COLAB:
    print("Đang tải validation_folds.csv về máy...")
    colab_files.download(str(fold_path))
else:
    print(f"Chạy local — artifact được giữ tại: {fold_path}")

Chạy local — artifact được giữ tại: /home/drago/projects/Machine-Learning-Deep-Learning/ISE_TRAINNING_TEST_23-8-2026/artifacts/validation_folds.csv


## Kết luận Phase 1

Phase 1 không cố làm điểm số cao; nó xây **thước đo đáng tin cậy**. Từ Phase 2, một thay đổi chỉ đáng giữ khi cải thiện OOF Macro F1 trên đúng các fold này. Nhờ vậy ta biết mức tăng đến từ model/feature chứ không phải do đổi cách chia dữ liệu.

Checklist hoàn thành:

- [x] Dùng 15 feature, loại `track_id` và target khỏi đầu vào.
- [x] Hash các dòng có feature giống nhau thành group.
- [x] Chia 3 fold bằng `StratifiedGroupKFold`, seed 42.
- [x] Mọi fold đủ 112 lớp.
- [x] Không có group leakage.
- [x] Lưu `artifacts/validation_folds.csv`.
- [x] Khóa hàm Macro F1 trên đủ 112 nhãn.

**Checkpoint:** dừng tại đây. Chỉ chuyển sang Phase 2 khi người dùng yêu cầu.